# Spatial phenology: SOS / POS / EOS rasters with `SpatialPhenologyAnalyzer`

Generate **per-pixel phenology rasters** (Start / Peak / End of Season and derived metrics) fully *server-side* on Google Earth Engine, using the `SpatialPhenologyAnalyzer` class from **ndvi2gif**.

Unlike `TimeSeriesAnalyzer` (phenology at a single **point**, downloading the series with `getInfo`), this computes the metrics for **every pixel** of the ROI and returns them as Earth Engine images.

## Three methods (all server-side)

| Method | Idea | Notes |
|---|---|---|
| `threshold` | SOS/EOS = first/last day-of-year above an amplitude fraction of the seasonal curve | Fast and robust |
| `derivative` | SOS/EOS = days of steepest increase / decrease between composites | Captures fast green-up |
| `harmonic` | Per-pixel **Fourier** regression -> smooth curve -> threshold extraction | Replaces the point-based double-logistic fit (not portable to GEE) |

## Two outputs

- **Option A** - one raster per year (`ee.ImageCollection`): study how phenology shifts over time.
- **Option B** - a single multi-year aggregate (`ee.Image`): the typical phenology of the area.

Output bands: `sos, pos, eos, los, amplitude, peak_value, baseline, growth_rate, senescence_rate` (SOS/POS/EOS/LOS in day-of-year).

> **Recommendation**: use `periods=12` (monthly) or `periods=24` (bi-monthly). With `periods=4` the class warns that the intra-annual resolution is too coarse.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import ee
import geemap

from ndvi2gif import NdviSeasonality, SpatialPhenologyAnalyzer

ee.Authenticate(auth_mode='localhost')
ee.Initialize(project='ee-digdgeografo')

## 1. ROI

A reproducible ROI over the **Isla Mayor** rice paddies (Donana, Seville), where the rice cycle is very marked: flooding/sowing in spring, peak greenness in summer, harvest and senescence in autumn. This makes SOS/POS/EOS easy to interpret.

To draw your own area instead, use `roi = Map.draw_last_feature.geometry()` (with `.geometry()` - `reduceRegion` does not accept a `Feature`).

In [3]:
roi = ee.Geometry.Rectangle([-6.27, 37.10, -6.13, 37.20])

Map = geemap.Map()
Map.centerObject(roi, 12)
Map.addLayer(roi, {'color': 'red'}, 'ROI - Isla Mayor')
Map

Map(center=[37.15000956278534, -6.2000000000002675], controls=(WidgetControl(options=['position', 'transparent…

## 2. Configure `NdviSeasonality` and `SpatialPhenologyAnalyzer`

Monthly NDVI composites (Sentinel-2). The `key` parameter sets the per-period reducer (`median`, `max`, `mean`, `min`, `sum`, `percentile`); phenology uses that series as its input.

In [5]:
processor = NdviSeasonality(
    roi=roi,
    periods=12,          # monthly -> good intra-annual resolution
    start_year=2019,
    end_year=2022,
    sat='S2',
    index='ndvi',
    key='percentile',         # try 'max', or key='percentile', percentile=90
    percentile=90
)

pheno = SpatialPhenologyAnalyzer(processor)

There we go again...
Applying cloud filter to Sentinel-2: max 20% cloud cover
Applying pixel-level cloud/shadow mask using SCL band
Using MODIS Terra + Aqua LST (maximum coverage)
Using all Sentinel-1 orbits (ascending + descending).
Applying S1 ARD preprocessing:
  - Speckle filter: REFINED_LEE
  - Terrain correction: True
  - Terrain model: VOLUME
Sentinel-2 collection configured with cloud filtering


## 3. Option A - one phenology raster per year

`extract_phenology_rasters` returns an `ee.ImageCollection` with one image per year (each carrying a `year` property).

In [6]:
yearly = pheno.extract_phenology_rasters(method='harmonic')

print('years:', yearly.size().getInfo())
print('bands:', ee.Image(yearly.first()).bandNames().getInfo())

doy_vis = {'min': 1, 'max': 365, 'palette': ['#2c7bb6', '#ffffbf', '#d7191c']}
los_vis = {'min': 30, 'max': 250, 'palette': ['white', 'green']}

img_2019 = ee.Image(yearly.first())
Map.addLayer(img_2019.select('sos'), doy_vis, 'SOS 2019')
Map.addLayer(img_2019.select('pos'), doy_vis, 'POS 2019', False)
Map.addLayer(img_2019.select('eos'), doy_vis, 'EOS 2019', False)
Map.addLayer(img_2019.select('los'), los_vis, 'LOS 2019 (days)', False)
Map

years: 4
bands: ['sos', 'pos', 'eos', 'los', 'amplitude', 'peak_value', 'baseline', 'growth_rate', 'senescence_rate']


Map(bottom=407890.0, center=[37.15000956278534, -6.2000000000002675], controls=(WidgetControl(options=['positi…

## 4. Option B - aggregate phenology map of the period

`phenology_summary` collapses all years into a single image with the median (or mean) of each metric: the typical phenology of the area.

In [7]:
summary = pheno.phenology_summary(method='harmonic', reducer='median')
print('summary bands:', summary.bandNames().getInfo())

Map.addLayer(summary.select('sos'), doy_vis, 'SOS median 2019-2022')
Map.addLayer(summary.select('eos'), doy_vis, 'EOS median 2019-2022', False)
Map

summary bands: ['sos', 'pos', 'eos', 'los', 'amplitude', 'peak_value', 'baseline', 'growth_rate', 'senescence_rate']


Map(bottom=407890.0, center=[37.15000956278534, -6.2000000000002675], controls=(WidgetControl(options=['positi…

## 5. Compare the three methods

Each method defines SOS differently, so differences are expected: `threshold` marks the amplitude-threshold crossing, `derivative` the moment of steepest increase (later in fast-greening crops), and `harmonic` works on the smoothed curve.

> **Performance note**: a `reduceRegion(...).getInfo()` over the whole ROI at fine scale may raise *"User memory limit exceeded"* in interactive mode. Mitigate with `tileScale` (spreads the memory) and a coarser scale for the numeric check. For the full-resolution product, use `export=True` (batch), which has a much larger budget.

In [8]:
geom = roi.geometry() if hasattr(roi, 'geometry') else roi

for m in ['threshold', 'derivative', 'harmonic']:
    s = pheno.phenology_summary(method=m).select('sos')
    mean_sos = s.reduceRegion(
        ee.Reducer.mean(), geom,
        scale=100,        # coarser, just for the numeric check
        maxPixels=1e9,
        bestEffort=True,
        tileScale=4       # spreads memory; raise to 8/16 if needed
    ).get('sos').getInfo()
    print(f'mean SOS ({m}): {mean_sos:.1f} (day of year)')

mean SOS (threshold): 104.5 (day of year)
mean SOS (derivative): 172.9 (day of year)
mean SOS (harmonic): 85.4 (day of year)


## 6. Export to GeoTIFF

Both entry points accept `export=True`, with two targets:

- `export_target='local'` (default): downloads a GeoTIFF to the working directory. Band names are embedded with **rasterio** (if installed), so QGIS shows `sos`, `pos`, `eos`, ... instead of `Band 1, 2, 3`.
- `export_target='drive'`: starts a batch task to Google Drive. Earth Engine preserves band names natively, and the memory budget is much larger - the right choice for big ROIs or fine scales.

The per-year collection exports one file per year; the summary exports a single file.

In [9]:
# Local GeoTIFF (named bands via rasterio)
pheno.phenology_summary(method='harmonic', reducer='median',
                        export=True, export_target='local', scale=20)

# Batch export to Google Drive (band names preserved by Earth Engine)
pheno.phenology_summary(method='harmonic', reducer='median',
                        export=True, export_target='drive',
                        drive_folder='ndvi2gif_phenology', scale=20)

# One GeoTIFF per year
# pheno.extract_phenology_rasters(method='harmonic', export=True,
#                                 export_target='drive',
#                                 drive_folder='ndvi2gif_phenology', scale=20)

Exporting S2_ndvi_phenology_harmonic_median_2019_2022.tif
Generating URL ...
Please wait ...
Data downloaded to /home/diego/git/Ndvi2Gif/examples_notebooks/S2_ndvi_phenology_harmonic_median_2019_2022.tif
Image have been exported
Starting Drive export task 'S2_ndvi_phenology_harmonic_median_2019_2022' (folder=ndvi2gif_phenology). Track it in the Earth Engine Tasks panel.


## Useful parameters

- `threshold_percentile` (50): amplitude fraction defining the season.
- `adaptive_threshold` (True): lower threshold for SOS, higher for EOS.
- `n_harmonics` (2): number of Fourier harmonics.
- `harmonic_step` (10): day step when rebuilding the smooth curve (smaller = finer, costlier).
- `min_observations` (5): minimum valid composites per pixel and year.
- `reducer` (`'median'`): `'median'` or `'mean'` for the multi-year aggregate.
- `export_target` (`'local'`), `drive_folder`, `crs` (`'EPSG:4326'`), `scale`.

In `NdviSeasonality`, the `key` (`median`/`max`/`percentile`/...) controls the per-period reducer of the input series.